# Marching Qubes Vertex Classification with EHands and QCrank
## Brandon Watanabe


In [51]:
# imports, part 1
import numpy as np
import matplotlib.pyplot as plt
import math
from math import pi
import random
import sys
from dotenv import load_dotenv
import os
from pathlib import Path
from collections import defaultdict

import qiskit
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.quantum_info import Statevector, Operator, Pauli
from qiskit.visualization import plot_histogram, plot_bloch_multivector, plot_distribution, array_to_latex
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.options.sampler_options import SamplerOptions

print(f"Qiskit version: {qiskit.__version__}")

Qiskit version: 2.3.0


In [52]:
# Adapted from Chris Pestano's QSobel
load_dotenv()
data_encoder_circuits_path = os.getenv('DATA_ENCODER_CIRCUITS_PATH')

if data_encoder_circuits_path is None:
    raise EnvironmentError("DATA_ENCODER_CIRCUITS_PATH environment variable is not set. Please set it to the path of data-encoder-circuits.")

print(f"Using DATA_ENCODER_CIRCUITS_PATH from environment variable: {data_encoder_circuits_path}")
circuits_path = data_encoder_circuits_path

# Add to sys.path if not already present
if circuits_path not in sys.path:
    sys.path.insert(0, circuits_path)

print(f"Added to sys.path: {circuits_path}")
print(f"Current working directory: {os.getcwd()}")

from datacircuits.ParametricQCrankV2 import ParametricQCrankV2 as QCrankV2, analyze_qcrank_residuals

Using DATA_ENCODER_CIRCUITS_PATH from environment variable: C:\Users\burne\Documents\SchoolWork\QC-Classes\References\repos\data-encoder-circuits
Added to sys.path: C:\Users\burne\Documents\SchoolWork\QC-Classes\References\repos\data-encoder-circuits
Current working directory: c:\Users\burne\Documents\SchoolWork\QC-Classes\Coding Projects\Marching-Qubes-Thesis\exploration_notebooks


In [53]:
sim = AerSimulator()

print(sim)
print(f"\nConfiguration: {sim.configuration()}")

if hasattr(sim, 'available_methods'):
    print(f"Available methods: {sim.available_methods()}")
else:
    print(sim.configuration())

if hasattr(sim, 'available_devices'):
    print(f"Available devices: {sim.available_devices()}")
else:
    print(sim.configuration())   

AerSimulator('aer_simulator')

Configuration: <qiskit_aer.backends.backendconfiguration.AerBackendConfiguration object at 0x0000015C1D4F9710>
Available methods: ('automatic', 'statevector', 'density_matrix', 'stabilizer', 'matrix_product_state', 'extended_stabilizer', 'unitary', 'superop')
Available devices: ('CPU',)


In [54]:
def run_sim_job(qc, n_shots = 2**8, verbose=False):
    """ 
    Run a quantum circuit on the simulator and return the counts.
    
    :param qc: QuantumCircuit to run
    :param n_shots: Number of shots to run the circuit for
    :param verbose: If True, print the circuit and counts

    :return: Counts of the measurement outcomes
    """

    transpiled_circuit = transpile(qc, sim)    
    job = sim.run(transpiled_circuit, shots=n_shots)
    result = job.result()
    
    # Get counts and output distribution
    counts = result.get_counts(transpiled_circuit)

    if verbose:
        print("Circuit before transpilation")
        display(qc.draw('mpl'))
        print("Circuit after transpilation")
        display(transpiled_circuit.draw('mpl', style="iqp"))

        print("Counts:")
        print(counts)
        display(plot_distribution(counts))

    return counts

In [55]:
def display_statevector(qc):
    """
    Display the statevector, unitary matrix, and Bloch sphere representation of a quantum circuit.

    :param qc: QuantumCircuit to analyze

    :return: Statevector of the quantum circuit
    """
    # Get Statevector
    state = Statevector.from_instruction(qc)

    # Display Statevector
    print("\nStatevector:")
    display(state.draw("latex"))

    matrix_form = np.array(state).reshape(-1, 1)
    print("\nStatevector as a matrix:")
    display(array_to_latex(matrix_form))
    
    # Display the Unitary Matrix
    print("Unitary matrix:")
    display(Operator(qc).draw("latex"))

    # Display Bloch Sphere
    display(plot_bloch_multivector(state))

    return state

In [56]:
def plot_residuals(actual, theory, title, x_label, y_label, legend):
    # plot residual
    plt.scatter(theory, actual, label=legend)
    # calculate slope of line of best fit
    coefficients = np.polyfit(theory, actual, 1)
    print(f"Slope of line of best fit: {coefficients[0]:.4f}")
    # plot line of best fit
    plt.plot(theory, np.poly1d(coefficients)(theory), color='red', label='Line of Best Fit')    

    plt.xlabel(x_label)    
    plt.ylabel(y_label)
    plt.title(title)
    plt.legend()

    plt.show()

In [ ]:
def ehands_addition(qc, q_a, q_b, weight, negation=False, verbose=False, barrier=False):
    """ 
    Perform an EHands addition of two qubits with a given weight and optional negation.

    :param qc: QuantumCircuit to which the addition will be applied
    :param q_a: Index of the first qubit
    :param q_b: Index of the second qubit
    :param weight: Weight for the addition (between 0 and 1)
    :param negation: If True, negate the second qubit before addition

    :return: QuantumCircuit with the eHANDS addition applied   
    """
    alpha = np.arccos(1 - 2 * weight)

    qc_add = QuantumCircuit(2)

    if barrier:
        qc_add.barrier()

    if negation:
        qc_add.x(1)

    qc_add.rz(pi/2, 1)
    qc_add.cx(0, 1)
    qc_add.ry(alpha/2, 0)
    qc_add.cx(1, 0) 
    qc_add.ry(-alpha/2, 0)

    if barrier:
        qc_add.barrier()

    if verbose:
        print("Addition Circuit:")
        display(qc_add.draw('mpl'))
    
    return qc.compose(qc_add, qubits=[q_a, q_b])


def ehands_mult(qc, q_a, q_b):
    """Product-with-memory (Fig. 1b): q_a keeps its value, q_b gets q_a * q_b."""
    qc.cx(q_a, q_b)
    qc.rz(pi / 2, q_b)
    return qc


def ehands_parity_flip(qc, q_a, q_b):
    """Parity flip between weighted sums (Fig. 1e)."""
    qc.h(q_b)
    qc.cz(q_a, q_b)
    return qc


def measure_z_ev(counts):
    n0 = counts.get("0", 0)
    n1 = counts.get("1", 0)
    total = n0 + n1
    return (n0 - n1) / total if total else 0.0


In [ ]:
# Test addition 
x0 = 0.9
x1 = 0.1
negation = True

qc1 = QuantumCircuit(2, 1)

qc1.ry(np.arccos(x0), 0)
qc1.ry(np.arccos(x1), 1)

# Equal weighting of inputs
w = 0.5 

qc1 = ehands_addition(qc1, 0, 1, w, negation=negation)

# Display the circuit and statevector
#state = display_statevector(qc1)
state = Statevector.from_instruction(qc1)

qc_meas = qc1.copy()
qc_meas.measure(0, 0)

# Run simulation
n_shots = 2**12
counts = run_sim_job(qc_meas, n_shots=n_shots, verbose=False)

# Compute actual EV from measurement
ev_actual = 0
n0 = counts.get('0', 0)
n1 = counts.get('1', 0)
ev_actual = (n0 - n1) / (n0 + n1)

# Compute theoretical EV using weighted sum
ev_theory = 0
ev_theory = state.expectation_value(Pauli('Z'), [0])

"""
if negation:
    ev_theory = w*x0 - (1-w)*x1
else:
    ev_theory = w*x0 + (1-w)*x1

print(f"Actual EV of qubit 0: {ev_actual:.4f}")
print(f"Theoretical EV (weighted sum): {ev_theory:.4f}")
print(f"EV from statevector: {state.expectation_value(Pauli('Z'), [0]):.4f}")
"""

print(f"Actual EV of qubit 0: {ev_actual:.4f}")
print(f"Theoretical EV (weighted sum): {ev_theory:.4f}")



Actual EV of qubit 0: 0.3691
Theoretical EV (weighted sum): 0.4000


In [ ]:
# Circuit for f(x) = 10*x - (1000/3)*x^3 = u - u^3/3  (Taylor term of arctan(u), u = 10*x)
# Uses only ehands_mult, ehands_addition, ehands_parity_flip (+ EVEN Ry encoding).

COEFF_X3 = -1.0 / 3.0   # coefficient of u^3 in u - u^3/3
SCALE_U = 10.0          # u = 10*x


def ehands_encode(qc, q, value):
    """EVEN-encode value in [-1, 1] on qubit q (Fig. 1a)."""
    qc.ry(np.arccos(np.clip(value, -1.0, 1.0)), q)


def apply_10x_minus_cubic(qc, x, q_u=0, q_out=6, q_anc=(7, 8)):
    """
    Compose f(x) = 10*x - (1000/3)*x^3 on the circuit using EHands operators.

    q_u must hold EVEN-encoded x before the call (or pass the same classical x used
    to prepare q_u). Ancillas q1..q6 and parity qubits q_anc are allocated here.

    The result is EVEN-encoded on q_out (default q6); read with Z expectation value.
    Requires |u| = |10*x| <= 1.
    """
    u = float(np.clip(SCALE_U * x, -1.0, 1.0))
    q_u2, q_u3 = 1, 2
    q_b3, q_b1, q_b2 = 3, 4, 6
    q_s0, q_s1 = 5, 4
    q_pf0, q_pf1 = q_anc

    # --- powers of u (q0 keeps u; q2 ends with u^3) ---
    ehands_encode(qc, q_u, u)
    ehands_encode(qc, q_u2, u)
    ehands_encode(qc, q_u3, u)
    qc = ehands_mult(qc, q_u, q_u2)     # q_u2 = u^2
    qc = ehands_mult(qc, q_u2, q_u3)    # q_u3 = u^3

    # --- coefficient registers for P(u) = u + COEFF_X3 * u^3 ---
    ehands_encode(qc, q_b3, COEFF_X3)  # a3 on b3 via Ry(arccos(a3))
    ehands_encode(qc, q_b1, 0.0)       # padding register (atan template)
    ehands_encode(qc, q_b2, 0.0)

    # couple powers to coefficient lines (product-with-memory)
    qc = ehands_mult(qc, q_u3, q_b3)
    qc = ehands_mult(qc, q_u2, q_b1)
    qc = ehands_mult(qc, q_u, q_s0)

    # weighted accumulation: w=1/2, 2/3, 1/4 with parity flips (EHands Fig. 2 / atan_poly_3)
    qc = ehands_addition(qc, q_s1, q_b3, 0.5, negation=False)
    qc = ehands_parity_flip(qc, q_s1, q_pf0)
    qc = ehands_addition(qc, q_s0, q_s1, 2.0 / 3.0, negation=False)
    qc = ehands_parity_flip(qc, q_s0, q_pf1)
    qc = ehands_addition(qc, q_b2, q_s0, 0.25, negation=False)

    return qc


# --- build and display circuit for one sample x ---
x = 0.08
u = SCALE_U * x
theory = SCALE_U * x - (1000.0 / 3.0) * x**3

qc_poly = QuantumCircuit(9, 1)
apply_10x_minus_cubic(qc_poly, x)

print(f"x = {x},  u = 10*x = {u:.4f}")
print(f"Classical f(x) = 10*x - (1000/3)*x^3 = {theory:.6f}")
print(f"Taylor check u - u^3/3 = {u - u**3/3:.6f}")

display(qc_poly.draw("mpl", fold=-1))

state = Statevector.from_instruction(qc_poly)
ev_out = float(state.expectation_value(Pauli("Z"), [6]).real)
print(f"Statevector EV on q6 = {ev_out:.6f}")

qc_meas = qc_poly.copy()
qc_meas.measure(6, 0)
counts = run_sim_job(qc_meas, n_shots=2**12, verbose=False)
ev_shots = measure_z_ev(counts)
print(f"Shot EV on q6 = {ev_shots:.6f}")